# Pipeline Flow

This notebook, `doc_executiveSummary.ipynb`, creates document-level summaries from OCR text. Its combined Markdown files (`document_summaries*_combined_stage2_results.md`) are read by `Admission_Eligibility_Checker_using_KB.ipynb` for applicant-level eligibility evaluation.

`doc_reconciliationAgent.ipynb` is an optional, separate path: it compares per-document JSON summaries from multiple model-output folders and writes reconciled JSON files and audit CSVs. The current `Admission_Eligibility_Checker_using_KB.ipynb` does not read those reconciled outputs; it reads the combined Markdown files produced here.

<audio controls>
  <source src="https://raw.githubusercontent.com/fair-vid/video-multimodal-pipeline/main/assets/audio/doc_executiveSummary.wav" type="audio/wav">
</audio>

### 📁 Pipeline Inputs & Outputs

To help track the flow of data through this notebook, here is a summary of the directories and file types involved in each stage:

*   **Input Directory (`image_text_dir`)**:
    *   **Path**: `/content/drive/MyDrive/colab-output/dream_applicant_application/{applicant_id}_{application_id}/documents_image_llm_text_glm-ocr`
    *   **Content**: Raw page-level text files (`.txt`) extracted from applicant document images using OCR (e.g., `Passport.pdf (page 1).txt`).
*   **Output Directory (`output_sub_dir_name`)**:
    *   **Path**: `/content/drive/MyDrive/colab-output/dream_applicant_application/{applicant_id}_{application_id}/document_summaries___glm-ocr_granite418b`
    *   **Content**:
        *   **Stage 1 & 2 JSONs**: Standardized, depersonalized, and policy-enriched metadata files (`.json`) for each individual document page.
        *   **Combined Audit Report**: A consolidated Markdown file (`*_combined_stage2_results.md`) located in the applicant's parent directory, merging all JSON summaries for easier human verification.

# 📄 Document-Level Executive Summarization for International Admissions
### Two-Stage Privacy-Preserving Abstraction via Local LLMs (🦙 Ollama)

---

This notebook implements the **document-level executive summarization stage** of the proposed AI-driven admissions evaluation pipeline.

At this point in the workflow, individual document pages have already been processed via OCR, reconstructed into coherent multi-page documents, and stored as structured textual representations.

The purpose of this notebook is to elevate document processing from raw OCR extraction to **decision-oriented abstraction**. To achieve high accuracy and contextual awareness, this is executed via a **Two-Stage Agentic Pipeline**:

1. **Stage 1: Initial Classification & Routing:** A general prompt scans the document to determine its taxonomy (e.g., transcript, diploma, passport) and issuing country.
2. **Stage 2: Policy-Aware Deep Dive:** Using the metadata from Stage 1, the system dynamically routes the document to a specialized, context-specific prompt (e.g., `TR_high_school_transcript`). This specialized agent synthesizes the text into a structured, policy-aware executive summary.

---

## 🔐 Built-In Personal Information Removal

Throughout both stages, the local agent is explicitly instructed to:
* Remove **all personal identifying information (PII)**.
* Omit names, IDs, addresses, signatures, and personal identifiers.
* Rewrite summaries generically using terms such as *“the applicant”* or *“the student”*.

The result is a **personalized yet anonymized JSON and Markdown summary** of each applicant’s document set—preserving academic and institutional relevance while eliminating identity exposure.

---

## 🛡️ Production Security Layer (Not Enabled in This Demo)

In real-world deployments, an additional security layer can be applied before any cloud interaction:
* A strict lexical filter removes all words not present in a general English dictionary.
* This guarantees that no unique identifiers, names, or rare tokens can leave the local environment.

Empirical testing shows that modern local LLMs (such as **IBM Granite 4.1** or **Google Gemma 3** via Ollama) are already highly reliable at removing personal information when guided by strict prompt constraints. For clarity and reproducibility, this Colab demonstration relies solely on the **local privacy-preserving agent**, without activating the additional lexical filtering layer.

---

## 🎯 Functional Objectives

Using a local large language model via Ollama, this notebook sequentially executes:

* **Taxonomy Mapping:** Consolidates multi-page content to classify the document type.
* **Dynamic Routing:** Selects the exact prompt needed based on document type and country ISO code.
* **Targeted Extraction:** Identifies authoritative dates, institutional information, and applies strict rule-based grade inclusion logic aligned with admissions policy.
* **Data Structuring:** Merges the deep-dive insights into a unified JSON artifact and generates a combined `.md` file for human auditing.

This stage reflects how advanced admissions systems reduce evaluator cognitive load by transforming heterogeneous documents into:

> **Standardized, policy-aware, anonymized executive summaries**  
> *Suitable for audit, comparison, and multimodal fusion.*

---

> **Methodological Note:**  
> In the full agent-based digital twin architecture described in the paper, executive document summarization operates in parallel with other analytical agents. The sequential execution shown here serves reproducibility and transparency within the Google Colab demonstration environment.

---



---



---



In [ ]:
import os
from google.colab import drive
import re

if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive is already mounted")

Mounting Google Drive...
Mounted at /content/drive


In [ ]:
!sudo apt update > /dev/null 2>&1 && sudo apt install -y pciutils lshw zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!pip install ollama -q

In [ ]:
model="granite4.1:8b" #  gemma4:e4b granite4.1:8b gemma3:12b
!ollama --version
!ollama run {model} > /dev/null 2>&1

ollama version is 0.34.4


# 🛠️ Configuration & Workflow Context

In this stage, two parameters define the scope and traceability of the agent processing:

*   **`image_text_dir`**: Specifies the source directory containing the OCR-extracted text that serves as the primary input for the summarization agents.
*   **`output_sub_dir_name`**: Defines the unique destination for the generated executive summaries.

**Strategic Importance:**
These two parameters together define a **single workflow instance**. In the complete architecture, multiple workflows (using different models or OCR engines) can run in parallel. Each workflow stores its results in distinct sub-directories, allowing a **Reconciliation Agent** in the next stage to compare, aggregate, and reconcile these diverse outputs into a final, high-confidence admissions profile.

In [ ]:
ocr_model_name = "glm-ocr"
image_text_dir = "documents_image_llm_text_"+ocr_model_name  #  documents_image_llm_text_glm-ocr   documents_image_llm_text_gemma4_31b      documents_image_llm_text_gemma3_12b_doc
output_sub_dir_name = "document_summaries"+"___"+ocr_model_name+"_" + re.sub(r'[^a-zA-Z0-9]', '', model)  #  document summaries_glm-ocr_gemma4_e4b

In [ ]:
#@title Clone the Prompt Repository
import os
import subprocess

repo_url = "https://github.com/fair-vid/admission_prompts.git"
repo_dir = "/content/admission_prompts"

# Clone if it doesn't exist, otherwise pull latest changes
if not os.path.exists(repo_dir):
    print("Cloning prompts repository...")
    subprocess.run(["git", "clone", repo_url, repo_dir])
else:
    print("Updating prompts repository...")
    subprocess.run(["git", "-C", repo_dir, "pull"])

prompt_folder = f"{repo_dir}/document_analysis"

Cloning prompts repository...


In [ ]:
#@title Load All Prompts into Memory

from pathlib import Path

def load_prompts(directory_path):
    prompts = {}
    path = Path(directory_path)

    # Read every text file in the directory
    for file_path in path.glob("*.txt"):
        prompt_name = file_path.stem  # e.g., 'transcript_prompt'
        with open(file_path, 'r', encoding='utf-8') as f:
            prompts[prompt_name] = f.read()

    print(f"Loaded {len(prompts)} prompts: {list(prompts.keys())}")
    return prompts

# Dictionary containing all your prompts
prompt_library = load_prompts(prompt_folder)

Loaded 14 prompts: ['undergraduate_degree', 'undergraduate_transcript_of_grades', 'Master_Admission_Eligibility_Checker_Prompt', 'TR_high_school_exam_transcript_of_grades', 'information_extractor_from_document_images_2', 'high_school_diploma_transcript_of_grades', 'CV', 'recommendation', 'document_classification_prompt', 'Bachelor_Admission_Eligibility_Checker_Prompt', 'English_language_proficiency_certificate', 'information_extractor_from_document_images', 'general_document_fallback_prompt', 'high_school_exam_transcript_of_grades']


In [ ]:
print(prompt_library.get('document_classification_prompt'))

TASK:
Read the provided DOCUMENT OCR TEXT and FILE NAME and output only valid JSON object about document.


PRELIMINARY OUTPUT JSON SCHEMA
{
  "document_type": "",
  "country_code": "",
  "issuing_country": "",
  "document_date": "",
  "executive_summary": "",
  "duration": "",
  "institution": "",
}


For "document_type", choose exactly ONE value from:
[
  "high_school_diploma",
  "high_school_diploma_transcript_of_grades",
  "high_school_exam_transcript_of_grades",
  "undergraduate_degree",
  "undergraduate_transcript_of_grades",
  "graduate_degree",
  "graduate_transcript_of_grades",
  "motivation_letter",
  "phd_diploma",
  "passport",
  "apostille",
  "personal_ID",
  "CV",
  "bank_statement",
  "recommendation",
  "English_language_proficiency_certificate",
  "other"
]


Rules:
- "issuing_country" must contain the country that issued the document or where the institution is located.
- "country_code" must contain the ISO 3166-1 alpha-2 country code, using exactly 2 uppercase lette

# 📂 Stage 1: Initial Document Classification & General Extraction

In this first stage, the agent performs a preliminary scan of the document text. The primary goals are:

1.  **Taxonomy Mapping**: Classifying the document into a specific category (e.g., *high_school_diploma*, *passport*, *transcript*).
2.  **Entity Extraction**: Identifying foundational metadata such as the issuing country (ISO code), date of issuance, and the name of the institution.
3.  **Contextual Awareness**: Preparing the metadata necessary to trigger the correct 'Deep Dive' prompts in Stage 2.

This stage acts as the **router** for the entire pipeline, ensuring that subsequent specialized agents receive the correct document context and file-specific identifiers.

In [ ]:
def get_prompt(doc_text, file_name):
    # 1. Fetch the classification prompt from the dictionary loaded from GitHub
    prompt_template = prompt_library.get('document_classification_prompt')

    # 2. Inject the document text and file name into the placeholders
    final_prompt = prompt_template.replace('{doc_text}', doc_text).replace('{file_name}', file_name)

    return final_prompt

In [ ]:
import os
import pandas as pd

drive_dir = "/content/drive/MyDrive/colab-output/dream_applicant_application/"


filter_file = os.path.join(drive_dir, "FILTERED_LIST_Applicant_Application_ids.tsv")

if os.path.exists(filter_file):
    print(f"Found filter file: {filter_file}")
    filter_df = pd.read_csv(filter_file, sep='\t')
    allowed_subdirs = set(filter_df['applicant_application'].astype(str).tolist())
    subdirs = [d for d in os.listdir(drive_dir) if os.path.isdir(os.path.join(drive_dir, d)) and d in allowed_subdirs]
    print(f"Filtered to {len(subdirs)} subdirectories based on TSV.")
else:
    print("No filter file found. Processing all subdirectories.")
    subdirs = [d for d in os.listdir(drive_dir) if os.path.isdir(os.path.join(drive_dir, d))]

print(subdirs)

No filter file found. Processing all subdirectories.
['0_0', '60811_46042', '29696_25487']


In [ ]:
from pathlib import Path

drive_dir = "/content/drive/MyDrive/colab-output/dream_applicant_application/"

# Dictionary to store files by subdirectory
files_by_subdir = {}

# Iterate through the subdirs list (which is now potentially filtered)
for subdir_name in subdirs:
    subdir = Path(drive_dir) / subdir_name
    if subdir.is_dir():
        # Check if text input directory exists
        documents_image_path = subdir / image_text_dir

        if documents_image_path.exists() and documents_image_path.is_dir():
            # Get all direct files in documents_image
            files = [f for f in documents_image_path.iterdir() if f.is_file()]
            files_by_subdir[subdir.name] = files
            print(f"Found {len(files)} files in {subdir.name}/{image_text_dir}/")

Found 0 files in 0_0/documents_image_llm_text_glm-ocr/
Found 5 files in 60811_46042/documents_image_llm_text_glm-ocr/
Found 7 files in 29696_25487/documents_image_llm_text_glm-ocr/


In [ ]:
import ollama
import base64
import json

def extract_with_ollama(prompt, model):

    try:
        response = ollama.generate(
            model=model,
            prompt=prompt,
            format='json'
        )

        return response["response"]

    except Exception as e:
        print(f"An error occurred: {e}")
        return None


In [ ]:
import os
import json
import re
from pathlib import Path

def extract_json_from_response(response_text):
    """
    Extract and validate JSON from response text.
    Returns: (is_valid_json, content, file_extension)
    """
    # First, try to find JSON within markdown code blocks
    json_pattern = r'```json\s*\n(.*?)\n```'
    match = re.search(json_pattern, response_text, re.DOTALL | re.IGNORECASE)

    if match:
        json_str = match.group(1).strip()
        try:
            # Validate JSON
            json.loads(json_str)
            return True, json_str, '.json'
        except json.JSONDecodeError:
            # Invalid JSON in code block, return as text
            return False, response_text, '.txt'

    # No markdown blocks found, try parsing the entire response as JSON
    try:
        json.loads(response_text)
        return True, response_text, '.json'
    except json.JSONDecodeError:
        # Not valid JSON, return as text
        return False, response_text, '.txt'


# Iterate through each subdirectory in files_by_subdir
for subdir_name, files in files_by_subdir.items():
    # Create the output directory for LLM text files
    subdir_path = Path(drive_dir) / subdir_name
    output_dir = subdir_path / output_sub_dir_name  # f"document summaries_{model.replace(':', '_').replace(' ', '_')}"
    os.makedirs(output_dir, exist_ok=True)

    print(f"\nProcessing {subdir_name} with {model}...")

    # Process each image file
    for i_file, image_file in enumerate(files):


        if (output_dir / f"{image_file.stem}.txt").exists()  or (output_dir / f"{image_file.stem}.json").exists() :# Skip if output file already exists
                print(f"        ⊘ Skipped (already exists): {image_file.name[:10]}")
                continue

        doc_text = open(image_file).read()  # Read document and get response
        # Pass both text and filename to the updated prompt function
        prompt = get_prompt(doc_text, image_file.name)

        print(f"\n {i_file}/{len(files)} {image_file.name} ")
        #print(f"{prompt}")

        response_text = extract_with_ollama(prompt, model)

        if response_text is not None:
            # Extract JSON and determine output format
            is_json, content, ext = extract_json_from_response(response_text)

            output_file = output_dir / f"{image_file.stem}{ext}"  # Create output file path with appropriate extension

            # Write to file
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(content)


Processing 0_0 with granite4.1:8b...

Processing 60811_46042 with granite4.1:8b...
        ⊘ Skipped (already exists): 46042_CV- 
        ⊘ Skipped (already exists): 46042_MOI 
        ⊘ Skipped (already exists): 46042_12th
        ⊘ Skipped (already exists): 46042_10th
        ⊘ Skipped (already exists): 46042_IMG-

Processing 29696_25487 with granite4.1:8b...

 0/7 Letter of Attestation.pdf (page 1).txt 

 1/7 Passport.pdf (page 1).txt 

 2/7 Letter of Recommendation.pdf (page 1).txt 

 3/7 Mekie Kamara CV 07072021.pdf (page 2).txt 

 4/7 Mekie Kamara CV 07072021.pdf (page 3).txt 

 5/7 Mekie Kamara CV 07072021.pdf (page 4).txt 

 6/7 Mekie Kamara CV 07072021.pdf (page 5).txt 


# 🔍 Stage 2: Document-Specific Deep Dive Summarization

In this stage, the pipeline moves from general classification to **targeted extraction**. Based on the `document_type` and `country_code` identified in Stage 1, the system selects a specialized prompt from the library to perform a 'Deep Dive'.

**Key features of this stage:**

1.  **Dynamic Prompt Selection**: The agent looks for a specific matching prompt (e.g., `TR_high_school_exam_transcript_of_grades`). If a country-specific version isn't available, it falls back to a general template for that document type.
2.  **Fallback Safety Net**: If no exact prompt for the document type is found, the system falls back to the generic prompt named `general_document_fallback_prompt`. This ensures the pipeline still produces a valid JSON summary for unknown or unhandled document categories.
3.  **Policy-Aware Extraction**: These specialized prompts are designed to look for specific values required by admissions policies, such as standardized test scores (e.g., TYT/AYT in Turkey) or specific credit systems.
4.  **JSON Enrichment**: The results of this deep dive are merged back into the original Stage 1 JSON object under the `document_specific_details` key. This creates a single, comprehensive, and structured record for each document.
5.  **Traceability**: The system tags each file with `_stage2_completed` and `_applied_prompt` to ensure auditability and prevent redundant processing.

By the end of Stage 2, the raw OCR text has been transformed into a highly structured, policy-relevant data artifact ready for holistic applicant evaluation.


In [ ]:
#@title 🚀 STAGE 2: Document-Specific Deep Dive Summarization

import json
import re
from pathlib import Path

# Recommended fallback prompt name for unknown or unsupported document types.
# Keep this name aligned with the corresponding .txt file in the prompt repo.
DEFAULT_STAGE2_PROMPT_KEY = "general_document_fallback_prompt"

# Iterate through each subdirectory in files_by_subdir again
for subdir_name, original_files in files_by_subdir.items():
    subdir_path = Path(drive_dir) / subdir_name
    output_dir = subdir_path / output_sub_dir_name

    if not output_dir.exists():
        continue

    print(f"\nProcessing Stage 2 for {subdir_name}...")

    # --- NEW: Initialize a list to hold the markdown content for this applicant ---
    md_entries = []

    # Process only .json files from Stage 1
    json_files = list(output_dir.glob("*.json"))

    for i_file, json_path in enumerate(json_files):
        # 1. Read the existing JSON created in Stage 1
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                existing_data = json.load(f)
        except json.JSONDecodeError:
            print(f"           (Invalid JSON): {json_path.name}")
            continue

        # Check if Stage 2 has already been applied to prevent duplicate API calls
        if existing_data.get("_stage2_completed"):
            print(f"  ⊘ Skipped (Stage 2 already done): {json_path.name[:15]}...")

            # --- NEW: Remember already processed files for the final .md file ---
            formatted_json = json.dumps(existing_data, indent=4, ensure_ascii=False)
            md_entries.append(f"# {json_path.stem}\n```json\n{formatted_json}\n```\n")
            continue

        # 2. Extract document_type
        doc_type = existing_data.get("document_type", "")
        if not doc_type:
            print(f"  ⊘ Skipped (No document_type in JSON): {json_path.name[:15]}...")
            continue

        # Normalize the document_type to match the prompt keys
        #doc_type_clean = re.sub(r'[\s\-]+', '_', doc_type.lower().strip())
        doc_type_clean = doc_type.strip()

        # Extract and normalize country_code (if it exists)
        country_code = existing_data.get("country_code", "")
        country_code_clean = country_code
        print(f"  Country Code: {country_code}")

        # 3. STRICT MATCHING against prompt library with fallback to a generic default
        prompt_key = None

        # 3a. Try country-specific prompt first (e.g., "TR_high_school_diploma")
        if country_code_clean:
            country_specific_key = f"{country_code_clean}_{doc_type_clean}"
            country_specific_key_prompt = f"{country_code_clean}_{doc_type_clean}_prompt"

            if country_specific_key in prompt_library:
                prompt_key = country_specific_key
            elif country_specific_key_prompt in prompt_library:
                prompt_key = country_specific_key_prompt

        # 3b. If no country-specific prompt was found, fall back to the general document type
        if not prompt_key:
            if doc_type_clean in prompt_library:
                prompt_key = doc_type_clean
            elif f"{doc_type_clean}_prompt" in prompt_library:
                prompt_key = f"{doc_type_clean}_prompt"

        # 3c. Final fallback: use a generic default prompt for unknown document types
        if not prompt_key:
            if DEFAULT_STAGE2_PROMPT_KEY in prompt_library:
                prompt_key = DEFAULT_STAGE2_PROMPT_KEY
                print(f"  ⚠️ No exact prompt found for '{doc_type}' ({country_code}). Using fallback: {prompt_key}")
            else:
                print(f"  ⊘ Skipped (No exact prompt for '{doc_type}' or '{country_code}' and no fallback found): {json_path.name}...")
                continue

        # 4. Fetch the original document text
        original_file = next((f for f in original_files if f.stem == json_path.stem), None)

        if not original_file or not original_file.exists():
            print(f"  ⊘ Original text not found for {json_path.name}")
            continue

        original_doc_text = open(original_file, 'r', encoding='utf-8').read()

        # 5. Inject original text into the strictly matched prompt
        prompt_template = prompt_library[prompt_key]
        final_prompt = prompt_template.replace('{doc_text}', original_doc_text)

        print(f"\n\n  → Running '{prompt_key}' for {json_path.name}... \n\n")

        # 6. Call Ollama for specific extraction
        response_text = extract_with_ollama(final_prompt, model)

        if response_text is not None:
            # Extract JSON from Stage 2 response
            is_json, content, ext = extract_json_from_response(response_text)

            if is_json:
                try:
                    new_data = json.loads(content)

                    # 7. Merge the new data into the existing Stage 1 dictionary
                    existing_data["document_specific_details"] = new_data
                    existing_data["_stage2_completed"] = True
                    existing_data["_applied_prompt"] = prompt_key

                    # 8. Save the combined JSON back to the exact same file
                    with open(json_path, 'w', encoding='utf-8') as f:
                        json.dump(existing_data, f, indent=4, ensure_ascii=False)

                    print("✅ Appended & Saved")

                    # --- NEW: Remember the newly processed JSON for the final .md file ---
                    formatted_json = json.dumps(existing_data, indent=4, ensure_ascii=False)
                    md_entries.append(f"# {json_path.stem}\n```json\n{formatted_json}\n```\n")

                except json.JSONDecodeError:
                    print("❌ Failed (Invalid JSON generated in Stage 2)")
            else:
                print("❌ Failed (No JSON found in Stage 2 output)")
        else:
            print("❌ Failed (Ollama error)")

    # --- NEW: Create the combined Markdown file in the applicant's main directory ---
    if md_entries:
        # Save to: ../dream_applicant_application/<subdir_name>/ ....md
        md_file_path = subdir_path / (output_sub_dir_name+"_combined_stage2_results.md")

        with open(md_file_path, 'w', encoding='utf-8') as md_file:
            # Optional: Add a main header at the very top of the file
            md_file.write(f"# All Summaries for Applicant: {subdir_name}\n\n")
            # Join all remembered file blocks with a separator
            md_file.write("\n---\n\n".join(md_entries))

        print(f"\n📄 Saved combined Markdown file to: {md_file_path}")

print("\n🎉 Stage 2 Complete! All specific insights have been appended to JSONs and combined into Markdown files.")



Processing Stage 2 for 0_0...

Processing Stage 2 for 60811_46042...
  ⊘ Skipped (Stage 2 already done): 46042_CV- Saif....
  ⊘ Skipped (Stage 2 already done): 46042_12th Onli...
  ⊘ Skipped (Stage 2 already done): 46042_10th cert...
  ⊘ Skipped (Stage 2 already done): 46042_MOI - Sai...
  ⊘ Skipped (Stage 2 already done): 46042_IMG-20260...

📄 Saved combined Markdown file to: /content/drive/MyDrive/colab-output/dream_applicant_application/60811_46042/document_summaries___glm-ocr_granite418b_combined_stage2_results.md

Processing Stage 2 for 29696_25487...
  Country Code: LR


  → Running 'recommendation' for Letter of Attestation.pdf (page 1).json... 


✅ Appended & Saved
  Country Code: LR
  ⚠️ No exact prompt found for 'passport' (LR). Using fallback: general_document_fallback_prompt


  → Running 'general_document_fallback_prompt' for Passport.pdf (page 1).json... 


✅ Appended & Saved
  Country Code: LR


  → Running 'recommendation' for Letter of Recommendation.pdf (page 1).json

---

## ⏭️ Next Step: Data Reconciliation

Now that we have successfully generated document summaries from multiple AI models, the next step is to merge them into a single, highly accurate "source of truth."

In the next notebook, we will use the **Document Reconciliation Agent** to:
* Compare the JSON outputs field-by-field.
* Use smart voting logic (Consensus, Majority Vote) to resolve any discrepancies.
* Generate a final master dataset and audit logs.

👉 [Click here to open the Document Reconciliation Agent](https://colab.research.google.com/drive/1T92JMD3UEdxf0viVcqQNXQP7nBSZHLPj?usp=sharing)







---

## ▶️ OR Next Step: Applicant-Level Scholarship Evaluation (Cloud Reasoning Agent)



At this stage, all applicant documents have been:

* reconstructed from page-level OCR outputs,
* semantically consolidated into structured executive summaries,
* strictly depersonalized (all personal identifying information removed),
* converted into standardized JSON artifacts suitable for downstream reasoning.

The next stage of the framework shifts from **document-level understanding** to **applicant-level strategic evaluation**.

In this step:

* All depersonalized document summaries for a single applicant are aggregated.
* A cloud-based reasoning agent (Google Gemini) evaluates the profile holistically.
* The agent estimates scholarship admission probability across three institutional tiers:

  * Top 10 global university
  * Around rank 100
  * Around rank 500
* The output is a structured, machine-readable JSON with probabilities and concise justifications.

Importantly, this cloud-based reasoning stage operates **only on anonymized, policy-aligned summaries**, preserving the privacy boundary established in this notebook.

You can run the next stage here:

**Applicant-Level Scholarship Probability Estimation (Gemini)**
👉 [https://colab.research.google.com/drive/1M4ozjZU_8OOigFIomONNEyBREeonoT7i?usp=sharing](https://colab.research.google.com/drive/1M4ozjZU_8OOigFIomONNEyBREeonoT7i?usp=sharing)

This final step completes the transformation from raw scanned documents to interpretable, probabilistic admissions signals within the transparent AI digital twin framework.

---

